# NB03-A — BM25 Index Building

**Purpose:** Build BM25 indexes for both corpora. This notebook runs on CPU only — no GPU quota consumed.

**Outputs → `crosslingual-rag-indexes` Kaggle Dataset:**
```
indexes/bm25_hotpotqa.pkl
indexes/bm25_wikipedia_id.pkl
```

**Est. runtime:** ~20–40 min (CPU). Run this first, let it finish, then run NB03-B separately for BGE-M3.

**Why split from NB03-B?** The original single-notebook session crashes after ~30 min due to browser memory pressure from holding the BGE-M3 model in GPU + the FAISS index in RAM simultaneously. BM25 indexing is pure CPU/pickle — it never touches the GPU and completes well within the session window.

## 0. Install Dependencies

In [ ]:
!pip install -q rank_bm25 PySastrawi

## 1. Imports & Setup

In [ ]:
import os
import json
import pickle
import time
from pathlib import Path
from tqdm import tqdm

from rank_bm25 import BM25Okapi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

print("All imports OK — CPU-only run, no CUDA needed.")

## 2. Paths & Directories

In [ ]:
# ── Input paths (from crosslingual-rag-data dataset) ──────────────────────────
HOTPOTQA_DOCS_PATH = "/kaggle/input/crosslingual-rag-data/hotpotqa_source_docs.jsonl"
WIKIPEDIA_ID_PATH  = "/kaggle/input/crosslingual-rag-data/wikipedia_id_corpus.jsonl"

# ── Output directories ─────────────────────────────────────────────────────────
WORKING_DIR = Path("/kaggle/working")
INDEX_DIR   = WORKING_DIR / "indexes"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", INDEX_DIR)

# Verify inputs exist
for label, path in [("HotpotQA docs", HOTPOTQA_DOCS_PATH),
                    ("Wikipedia ID",   WIKIPEDIA_ID_PATH)]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1e6 if exists else 0
    print(f"  {label}: {'✓' if exists else '✗ MISSING'}  ({size:.1f} MB)")

## 3. Helpers

In [ ]:
def load_jsonl(path):
    """Load a JSONL file into a list of dicts."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def extract_text(record):
    """
    Extract a single text string from a document record.
    Handles different JSONL schemas from NB01:
      - {"text": ...}
      - {"content": ...}
      - {"title": ..., "text": ...} → "title. text"
    Returns empty string if no usable field found.
    """
    title = record.get("title", "").strip()
    text  = record.get("text", record.get("content", "")).strip()
    if title and text:
        return f"{title}. {text}"
    return text or title


print("Helpers defined.")

## 4. Sastrawi Preprocessor

In [ ]:
# Sastrawi for Indonesian-aware stemming + stopword removal.
# Used for BM25 tokenization of BOTH corpora.
# HotpotQA docs are English — Sastrawi will be suboptimal, but this matches
# the baseline paper's approach. The asymmetry (Sastrawi degrades EN-query BM25)
# is exactly what NB07 Δgap measurement will demonstrate empirically.

stemmer_factory  = StemmerFactory()
stopword_factory = StopWordRemoverFactory()
stemmer   = stemmer_factory.create_stemmer()
stopwords = set(stopword_factory.get_stop_words())


def sastrawi_tokenize(text):
    """Tokenize text using Sastrawi stemmer + stopword removal."""
    text   = text.lower()
    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords]
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens


# Quick test
test = "Universitas Indonesia adalah universitas terbaik di Jakarta"
print(f"Input:  {test}")
print(f"Tokens: {sastrawi_tokenize(test)}")

## 5. Load Corpora

In [ ]:
print("Loading HotpotQA source docs...")
hotpotqa_docs  = load_jsonl(HOTPOTQA_DOCS_PATH)
hotpotqa_texts = [extract_text(d) for d in hotpotqa_docs]
hotpotqa_ids   = [d.get("id", str(i)) for i, d in enumerate(hotpotqa_docs)]
print(f"  Loaded {len(hotpotqa_texts)} docs")
print(f"  Sample: {hotpotqa_texts[0][:120]}...")

empty = sum(1 for t in hotpotqa_texts if not t.strip())
if empty:
    print(f"  ⚠ {empty} empty texts — check extract_text()")

print()
print("Loading Wikipedia ID corpus...")
wiki_docs  = load_jsonl(WIKIPEDIA_ID_PATH)
wiki_texts = [extract_text(d) for d in wiki_docs]
wiki_ids   = [d.get("id", str(i)) for i, d in enumerate(wiki_docs)]
print(f"  Loaded {len(wiki_texts)} docs")
print(f"  Sample: {wiki_texts[0][:120]}...")

empty = sum(1 for t in wiki_texts if not t.strip())
if empty:
    print(f"  ⚠ {empty} empty texts — check extract_text()")

## 6. Build BM25 Indexes

In [ ]:
def build_bm25_index(texts, ids, corpus_name, out_path):
    """
    Tokenize corpus and build BM25Okapi index.
    Saves both the BM25 object and doc_ids/texts to a single pickle.
    Pickle schema: {"bm25": BM25Okapi, "doc_ids": list[str], "texts": list[str]}
    """
    print(f"[BM25] Tokenizing {corpus_name} ({len(texts)} docs)...")
    t0 = time.time()
    tokenized = [sastrawi_tokenize(t) for t in tqdm(texts, desc="Tokenizing")]
    print(f"  Tokenization done in {time.time() - t0:.1f}s")

    print(f"[BM25] Building index...")
    t0 = time.time()
    bm25 = BM25Okapi(tokenized)
    print(f"  Index built in {time.time() - t0:.1f}s")

    payload = {"bm25": bm25, "doc_ids": ids, "texts": texts}
    out_path = Path(out_path)
    with open(out_path, "wb") as f:
        pickle.dump(payload, f)
    size_mb = out_path.stat().st_size / 1e6
    print(f"  ✓ Saved to {out_path} ({size_mb:.1f} MB)")
    return bm25


print("=" * 55)
print("BM25 Index: HotpotQA source docs")
print("=" * 55)
bm25_hotpotqa = build_bm25_index(
    hotpotqa_texts, hotpotqa_ids,
    "HotpotQA source docs",
    INDEX_DIR / "bm25_hotpotqa.pkl"
)

print()
print("=" * 55)
print("BM25 Index: Wikipedia ID corpus")
print("=" * 55)
bm25_wikipedia = build_bm25_index(
    wiki_texts, wiki_ids,
    "Wikipedia ID corpus",
    INDEX_DIR / "bm25_wikipedia_id.pkl"
)

## 7. Verify Artifacts

In [ ]:
required = [
    INDEX_DIR / "bm25_hotpotqa.pkl",
    INDEX_DIR / "bm25_wikipedia_id.pkl",
]

all_ok = True
print("Artifact check:")
for path in required:
    exists = path.exists()
    size   = path.stat().st_size / 1e6 if exists else 0
    status = f"✓ ({size:.1f} MB)" if exists else "✗ MISSING"
    print(f"  {status}  {path.name}")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("Both BM25 indexes present. Ready to upload to crosslingual-rag-indexes.")
else:
    print("⚠ Missing artifacts — rerun the indexing cells above.")

## 8. Persist to Kaggle Dataset (`crosslingual-rag-indexes`)

In [ ]:
import shutil

# Copy BM25 pickles to /kaggle/working/ top-level for easy upload
for pkl_name in ["bm25_hotpotqa.pkl", "bm25_wikipedia_id.pkl"]:
    src  = INDEX_DIR / pkl_name
    dest = WORKING_DIR / pkl_name
    if src != dest:
        shutil.copy(src, dest)
    size_mb = dest.stat().st_size / 1e6
    print(f"  Staged {pkl_name} ({size_mb:.1f} MB) → {dest}")

print()
print("Next steps:")
print("  1. Go to Notebook → Save & Run All  (commits output files)")
print("  2. Upload bm25_hotpotqa.pkl and bm25_wikipedia_id.pkl to the")
print("     'crosslingual-rag-indexes' Kaggle Dataset via its UI.")
print("  3. Run NB03-B in a fresh session for BGE-M3 embedding.")

## Quick-Load Snippet for Downstream Notebooks

Copy this into any notebook that needs the BM25 indexes:

```python
import pickle

with open("/kaggle/input/crosslingual-rag-indexes/bm25_hotpotqa.pkl", "rb") as f:
    payload = pickle.load(f)
    bm25_hotpotqa = payload["bm25"]
    hotpotqa_ids  = payload["doc_ids"]
    hotpotqa_texts = payload["texts"]

with open("/kaggle/input/crosslingual-rag-indexes/bm25_wikipedia_id.pkl", "rb") as f:
    payload = pickle.load(f)
    bm25_wiki  = payload["bm25"]
    wiki_ids   = payload["doc_ids"]
    wiki_texts = payload["texts"]

print("BM25 indexes loaded.")
```